# GEAP Agent Platform — Build → Deploy → Register (SDK-First Companion)

This is the **companion** to `src/eval/demo/evaluation_sdk_demo.ipynb`. That notebook teaches the Quality Flywheel (evaluation); this one teaches **everything before it** — build a multi-agent system with ADK + MCP tools, run it, deploy it to Agent Engine, register it to Gemini Enterprise, and route requests across models by cost. It stays **ADK / L1-SDK-first**: every phase calls an ADK (`google.adk`) or `vertexai` object, or a thin repo helper, rather than reimplementing anything.

**Legend.** ✅ = runs live here (in-process, or a cheap/read call). 🔒 = shown-but-guarded: a mutating or expensive infra step (Cloud Run / Agent Engine deploy, GE publish) prints the exact call and only executes when you set an env flag (`GEAP_RUN_DEPLOY=1`, `GEAP_PUBLISH=1`) — mirroring the eval notebook's `GEAP_RUN_GEPA=1` idiom. 🔧 marks acknowledged custom/infra code (not an ADK object).

> Pinned: `google-cloud-aiplatform[adk,agent_engines] 1.162`. Headless twins: `bash scripts/deploy_all.sh` (build + deploy) and `python scripts/publish_agents_to_ge.py` (register). When you're ready to measure quality, continue in **`evaluation_sdk_demo.ipynb`**.

## Setup

In [1]:
import os
# Run from the repo root so `from src...` imports and relative paths resolve.
for _ in range(6):
    if os.path.exists("src/config.py"):
        break
    os.chdir("..")

import vertexai
from vertexai import agent_engines
from src.config import GCP_PROJECT_ID, GCP_REGION, AGENT_ENGINE_ID

vertexai.init(project=GCP_PROJECT_ID, location=GCP_REGION)
AGENT_RESOURCE = f"projects/{GCP_PROJECT_ID}/locations/{GCP_REGION}/reasoningEngines/{AGENT_ENGINE_ID}"

# 🔒 guards — mutating/expensive steps run only when you opt in.
RUN_DEPLOY = os.environ.get("GEAP_RUN_DEPLOY") == "1"
PUBLISH = os.environ.get("GEAP_PUBLISH") == "1"

def agent_answer(prompt: str, user_id: str = "platform-demo") -> str:
    """🔧 Query the deployed coordinator (Agent Engine stream_query) and return its final text."""
    engine = agent_engines.get(AGENT_RESOURCE)
    texts = []
    for event in engine.stream_query(message=prompt, user_id=user_id):
        for part in ((event.get("content") or {}).get("parts") or []):
            if part.get("text"):
                texts.append(part["text"])
    return "\n".join(texts).strip() or "(no response)"

print("project:", GCP_PROJECT_ID, "| region:", GCP_REGION)
print("coordinator engine:", AGENT_RESOURCE)
print("guards -> RUN_DEPLOY:", RUN_DEPLOY, "| PUBLISH:", PUBLISH)

project: wortz-project-352116 | region: us-central1
coordinator engine: projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024
guards -> RUN_DEPLOY: False | PUBLISH: False


## Phase 1 — MCP tool servers (the agent's hands)
📖 [Workshop guide · MCP servers](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

Agents act through **tools**. We build them as **FastMCP** servers (`@mcp.tool()`) — one tool per Python function — deployed to Cloud Run over StreamableHTTP. Here we import the real `search-mcp` server and call its tools in-process against its mock DB (no network).

In [2]:
from src.mcp_servers.search import server as search_mcp

flights = search_mcp.search_flights("SFO", "JFK")
hotels = search_mcp.search_hotels("New York", max_price=350)
print(f"search_flights('SFO','JFK') -> {len(flights)} results; first: {flights[0]}")
print(f"search_hotels('New York', max_price=350) -> {len(hotels)} results; first: {hotels[0]}")
print("\nMCP server:", search_mcp.mcp.name, "| tools: search_flights, search_hotels")

# 🔧 Deploy the three MCP servers to Cloud Run (guarded — one command):
print("\n[deploy] bash scripts/deploy_all.sh   # builds + deploys search/booking/expense MCP + coordinator")

search_flights('SFO','JFK') -> 2 results; first: {'id': 'FL001', 'airline': 'United', 'origin': 'SFO', 'destination': 'JFK', 'date': '2026-06-15', 'price': 450.0, 'departure': '08:00', 'arrival': '16:30'}
search_hotels('New York', max_price=350) -> 2 results; first: {'id': 'HT001', 'name': 'Grand Hyatt New York', 'city': 'New York', 'price_per_night': 320.0, 'rating': 4.5, 'available_from': '2026-06-01', 'available_to': '2026-12-31'}

MCP server: search-mcp | tools: search_flights, search_hotels

[deploy] bash scripts/deploy_all.sh   # builds + deploys search/booking/expense MCP + coordinator


## Phase 2 — Build ADK agents (coordinator + specialists)
📖 [ADK LlmAgent](https://google.github.io/adk-docs/agents/llm-agents/) · [Workshop guide](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

An agent is `LlmAgent(model, instruction, tools)`. The **coordinator** delegates to specialist `sub_agents` (travel, expense) and also calls search tools directly. `resolve_model()` wraps Gemini 3.x / Claude for Vertex's `global` endpoint. (Building the agents connects to the Agent Registry with a direct-URL fallback, so this runs offline.)

In [3]:
from src.agents.coordinator_agent import coordinator_agent

print("root agent:", coordinator_agent.name)
print("sub_agents:", [a.name for a in coordinator_agent.sub_agents])
print("tools:", [type(t).__name__ for t in coordinator_agent.tools])
print("model (resolved):", type(coordinator_agent.model).__name__)
print("\ninstruction (first 220 chars):\n", coordinator_agent.instruction[:220], "...")

/home/admin_jwortz_altostrat_com/geap-tour/.venv/lib/python3.13/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
/home/admin_jwortz_altostrat_com/geap-tour/src/agents/_shared.py:29: UserWarning: [GEMINI_VIA_LITELLM] vertex_ai/gemini-3.5-flash: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='vertex_ai/gemini-3.5-flash') with Gemini(model='gemini-3.5-flash'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  return LiteLlm(model=model_str, vertex_location="global")


root agent: coordinator_agent
sub_agents: ['travel_agent', 'expense_agent']
tools: ['McpToolset', 'PreloadMemoryTool']
model (resolved): LiteLlm

instruction (first 220 chars):
 You are a corporate assistant coordinator. Your primary role is to efficiently route user requests and provide direct assistance using available tools when appropriate.

1. Direct Tool Usage (Your Primary Action):
   - F ...


/home/admin_jwortz_altostrat_com/geap-tour/src/agents/_shared.py:29: UserWarning: [GEMINI_VIA_LITELLM] vertex_ai/gemini-3.5-flash: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='vertex_ai/gemini-3.5-flash') with Gemini(model='gemini-3.5-flash'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  return LiteLlm(model=model_str, vertex_location="global")


## Phase 3 — Run it: multi-agent coordination
📖 [Agent Engine · query](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

The coordinator is already deployed to Agent Engine. Send a real prompt and watch it **route**: it has only the search tools itself, so an expense-policy question is delegated to `expense_agent` (bookings go to `travel_agent`), while simple searches it answers directly.

In [4]:
try:
    q = "Check if a $50 meal expense is within policy"
    print("Q:", q, "\nA:", agent_answer(q)[:600])
except Exception as e:
    print(f"(live query needs a reachable deployed coordinator + creds — {type(e).__name__}: {e})")

Q: Check if a $50 meal expense is within policy 
A: The $50 meal expense is within the policy limit of $75.


## Phase 4 — Deploy to Agent Engine (deploy) — 🔧 guarded
📖 [Deploy agents](https://github.com/jswortz/geap-tour/blob/main/docs/workshop_guide.md)

`agent_engines.create/update` packages the agent (`extra_packages=["src"]`), its `requirements`, telemetry `env_vars`, identity, and gateway config. We show the real config `deploy_agents.py` builds; the create/update runs only with `GEAP_RUN_DEPLOY=1`.

In [5]:
from src.deploy.deploy_agents import _build_config, run_deploy

config = _build_config(coordinator_agent)
print("deploy config keys:", sorted(config))
print("  extra_packages:", config["extra_packages"], "| identity:", config.get("identity_type", "default"))
print("  telemetry:", config["env_vars"]["GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY"],
      "| tiers:", [config["env_vars"][k] for k in ("LITE_MODEL","FLASH_MODEL","OPUS_MODEL")])

if RUN_DEPLOY:
    print("\n[deploying] run_deploy('coordinator', update=True) ...")
    print(run_deploy("coordinator", update=True))
else:
    print("\n🔒 skipped live deploy (set GEAP_RUN_DEPLOY=1). Existing engine:", AGENT_ENGINE_ID)
    print("   headless: uv run python -m src.deploy.deploy_agents coordinator --update")

  Identity: default (set ENABLE_AGENT_IDENTITY=1 to enable)
  Gateway: disabled (set ENABLE_AGENT_GATEWAY=1 to enable)
deploy config keys: ['display_name', 'env_vars', 'extra_packages', 'labels', 'requirements', 'staging_bucket']
  extra_packages: ['src'] | identity: default
  telemetry: true | tiers: ['gemini-3.1-flash-lite', 'gemini-3.5-flash', 'claude-opus-4-6']

🔒 skipped live deploy (set GEAP_RUN_DEPLOY=1). Existing engine: 5895016748914049024
   headless: uv run python -m src.deploy.deploy_agents coordinator --update


## Phase 5 — Register to Gemini Enterprise (publish) — 🔧 guarded
📖 [Publishing agents to Gemini Enterprise](https://github.com/jswortz/geap-tour/blob/main/docs/publishing_agents_to_gemini_enterprise.md)

Registration wraps the deployed reasoning engine in a Discovery Engine **`adkAgentDefinition` → `provisionedReasoningEngine`** and upserts it into your GE app (idempotent by engine resource). We show the payload shape; the real POST/PATCH runs only with `GEAP_PUBLISH=1`.

In [6]:
import json
# 🔧 The registration payload (mirrors scripts/publish_agents_to_ge.py::publish_one):
payload = {
    "displayName": "GEAP Coordinator",
    "description": "Corporate travel + expense multi-agent assistant.",
    "adkAgentDefinition": {
        "provisionedReasoningEngine": {"reasoningEngine": AGENT_RESOURCE},
    },
}
print(json.dumps(payload, indent=2))

if PUBLISH:
    import subprocess
    out = subprocess.run(["python", "scripts/publish_agents_to_ge.py", "coordinator"],
                         capture_output=True, text=True)
    print(out.stdout or out.stderr)
else:
    print("\n🔒 skipped live publish (set GEAP_PUBLISH=1). Preview all payloads with:")
    print("   python scripts/publish_agents_to_ge.py --dry-run")

{
  "displayName": "GEAP Coordinator",
  "description": "Corporate travel + expense multi-agent assistant.",
  "adkAgentDefinition": {
    "provisionedReasoningEngine": {
      "reasoningEngine": "projects/wortz-project-352116/locations/us-central1/reasoningEngines/5895016748914049024"
    }
  }
}

🔒 skipped live publish (set GEAP_PUBLISH=1). Preview all payloads with:
   python scripts/publish_agents_to_ge.py --dry-run


## Phase 6 — Complexity routing + multi-model cost (multi-model-router)
📖 [Multi-model cost comparison](https://github.com/jswortz/geap-tour/blob/main/docs/multi_model_cost_comparison.md)

A Flash-Lite micro-judge scores each prompt 0–1 and routes to the cheapest capable tier (lite → flash → sonnet → pro → opus). Below: live classification of three prompts, then the router-vs-all-Opus cost model.

In [7]:
from src.router.complexity import classify_complexity, score_to_model_tier
from app.cost_model import build_accrual

prompts = [
    "What's the meal per-diem?",
    "Compare the cheapest SFO-NYC flights by airline",
    "Plan a 5-day Tokyo trip for 4 with flights, hotels, and a per-diem budget",
]
try:
    for p in prompts:
        r = await classify_complexity(p)
        print(f"  score={r.score:.2f} -> {score_to_model_tier(r.score):6s} | {p[:52]}")
except Exception as e:
    print(f"(live classification needs GCP creds — {type(e).__name__}: {e})")

acc = build_accrual()
print(f"\nRouter cost model over {len(acc.steps)} prompts:")
print(f"  Smart router: ${acc.router_total:.4f}   all-Opus: ${acc.baseline_total:.4f}   savings: {acc.savings_pct:.0f}%")
for tier, n in acc.tier_counts.items():
    print(f"    {tier:6s} x{n}: ${acc.tier_cost[tier]:.4f}")

/home/admin_jwortz_altostrat_com/geap-tour/src/router/agents.py:38: UserWarning: [GEMINI_VIA_LITELLM] vertex_ai/gemini-3.1-flash-lite: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='vertex_ai/gemini-3.1-flash-lite') with Gemini(model='gemini-3.1-flash-lite'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this warning.
  return LiteLlm(model=model_str, vertex_location="global")
/home/admin_jwortz_altostrat_com/geap-tour/src/router/agents.py:38: UserWarning: [GEMINI_VIA_LITELLM] vertex_ai/gemini-3.5-flash: You are using Gemini via LiteLLM. For better performance, reliability, and access to latest features, consider using Gemini directly through ADK's native Gemini integration. Replace LiteLlm(model='vertex_ai/gemini-3.5-flash') with Gemini(model='gemini-3.5-flash'). Set ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS=true to suppress this 

  score=0.10 -> lite   | What's the meal per-diem?


  score=0.40 -> flash  | Compare the cheapest SFO-NYC flights by airline


  score=0.85 -> opus   | Plan a 5-day Tokyo trip for 4 with flights, hotels, 

Router cost model over 12 prompts:
  Smart router: $0.1639   all-Opus: $0.4860   savings: 66%
    Lite   x5: $0.0009
    Flash  x3: $0.0010
    Opus   x4: $0.1620


## Recap — ADK / L1-native vs 🔧 infra/custom

**ADK / vertexai-native:**
- `@mcp.tool()` FastMCP servers (`src/mcp_servers/*`) — the agent's tools
- `LlmAgent(model, instruction, tools, sub_agents)` — coordinator + specialists (`src/agents/*`)
- `agent_engines.get(...).stream_query(...)` — run the deployed agent
- `agent_engines.create/update` — deploy to Agent Engine (`src/deploy/deploy_agents.py`)
- `classify_complexity` → `score_to_model_tier` — multi-model routing (`src/router/*`)

**🔧 Acknowledged infra/custom:**
- Cloud Run deploy of MCP servers (`scripts/deploy_all.sh`) — 🔒 guarded
- Agent Engine deploy (`run_deploy`, `GEAP_RUN_DEPLOY=1`) — 🔒 guarded
- GE registration via Discovery Engine `adkAgentDefinition` (`scripts/publish_agents_to_ge.py`, `GEAP_PUBLISH=1`) — 🔒 guarded
- deterministic cost model (`app/cost_model.py`)

**Next:** measure and improve quality in **`src/eval/demo/evaluation_sdk_demo.ipynb`** (the Quality Flywheel). Headless twins: `bash scripts/deploy_all.sh`, `python scripts/publish_agents_to_ge.py`.